# Chapter 10 live coding + homework — MLPs in PyTorch

**Group members (name and Student id)**:

**Reference:** Aurélien Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch*, Ch. 10, “Building an Image Classifier with PyTorch”.

This notebook contains two parts:

1. **Live skeleton:** complete the Fashion-MNIST MLP introduced in class.
2. **Homework extension:** run controlled experiments on Fashion-MNIST and then design a systematic Optuna study on CIFAR-10.

> Keep the **test set untouched** until the final evaluation of a selected model. Hyperparameters must be chosen from training/validation results only.


In [ ]:
import copy
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms.v2 as T
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("device:", device)


## Part I — Complete the live Fashion-MNIST notebook

The first requirement of the homework is simple: **complete all live TODOs** below and make sure the notebook runs from top to bottom.

The objective is not just to obtain a good accuracy. You should be able to explain the role of the model, loss, gradients, optimizer, train/eval modes, logits, and probabilities.


### 0. Fashion-MNIST data — prepared for you

Images have shape `[C, H, W] = [1, 28, 28]`; a mini-batch has shape `[B, 1, 28, 28]`.


In [ ]:
to_tensor = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=to_tensor
)
test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=to_tensor
)

generator = torch.Generator().manual_seed(SEED)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000], generator=generator
)

BATCH_SIZE = 32
pin = device.type == "cuda"

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=pin)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin)

class_names = train_and_valid_data.classes
print(f"train={len(train_data)}, valid={len(valid_data)}, test={len(test_data)}")
print(class_names)


### 1. Inspect one mini-batch — reason before coding

Before running the cell, predict:

- `X_batch.shape` and `y_batch.shape`;
- the number of features after `nn.Flatten()`;
- the number of output units required by the classifier.


In [ ]:
X_batch, y_batch = next(iter(train_loader))
print("X_batch:", X_batch.shape, X_batch.dtype)
print("y_batch:", y_batch.shape, y_batch.dtype)
print("first labels:", y_batch[:8].tolist())

fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, image, label in zip(axes, X_batch[:6], y_batch[:6]):
    ax.imshow(image.squeeze(0), cmap="gray")
    ax.set_title(class_names[label])
    ax.axis("off")
plt.tight_layout()


### 2. Build the classifier

Complete the architecture

`[1, 28, 28] → 784 → 300 → 100 → 10`.

Do **not** add Softmax to the final layer.


In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()

        # TODO 1 — build the MLP with Flatten, Linear and ReLU layers.
        # self.mlp = nn.Sequential(
        #     ...
        # )

    def forward(self, X):
        # TODO 2 — return the logits.
        pass


torch.manual_seed(SEED)

# TODO 3 — instantiate the model and move it to `device`.
# model = ...


### 3. Loss, forward pass and optimizer

Use `nn.CrossEntropyLoss()`. It expects **logits** and integer class indices.


In [ ]:
# TODO 4 — create the loss.
# criterion = ...

X_batch, y_batch = next(iter(train_loader))
X_batch, y_batch = X_batch.to(device), y_batch.to(device)

# TODO 5 — compute logits and loss.
# logits = ...
# loss = ...

# TODO 6 — create an SGD optimizer.
# optimizer = torch.optim.SGD(...)


### 4. One optimization step

Put the essential operations in the correct order: forward pass, loss, backpropagation, parameter update, gradient reset.


In [ ]:
X_batch, y_batch = next(iter(train_loader))
X_batch, y_batch = X_batch.to(device), y_batch.to(device)

# TODO 7 — complete one optimization step.
# ...


### 5. Mini-batch training and validation

Complete `train_one_epoch`. The validation helper is already provided.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device=device):
    # TODO 8 — switch to training mode.
    # ...
    running_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # TODO 9 — complete the training operations.
        # ...

        running_loss += loss.item() * X_batch.size(0)

    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion=None, device=device):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        logits = model(X_batch)

        if criterion is not None:
            running_loss += criterion(logits, y_batch).item() * X_batch.size(0)

        correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total += y_batch.numel()

    avg_loss = running_loss / total if criterion is not None else None
    return {"loss": avg_loss, "accuracy": correct / total}


### 6. Train the baseline

Train for a few epochs and store the history. Your Fashion-MNIST experiments later in the notebook will use this result as the **baseline**.


In [ ]:
N_EPOCHS = 10

# TODO 10 — run training and validation, storing one dictionary per epoch.
# baseline_history = []
# for epoch in range(N_EPOCHS):
#     ...

# TODO 11 — report final validation accuracy.


### 7. Logits, predictions, probabilities, and final test-set evaluation

Complete the inference cell. Explain in one or two sentences why Softmax is useful here but not required inside the model during training.

Then evaluate the **fixed baseline model** on the test set. At this point the model architecture and training choices must already be fixed: the test set must **not** be used to tune hyperparameters or select among competing models.

Report:

- test loss;
- test accuracy;
- the difference between validation and test accuracy;
- one short interpretation of whether the two estimates are reasonably consistent.


In [ ]:
# TODO 12 — select six validation examples and compute logits.
# TODO 13 — obtain predicted classes with argmax.
# TODO 14 — convert logits to probabilities with softmax.

# TODO 15 — evaluate the FIXED baseline model on the test set.
# Hint: reuse the evaluate(...) helper and pass criterion to obtain both loss and accuracy.
# test_metrics = ...

# TODO 16 — print test loss and test accuracy.
# print(...)

# TODO 17 — compare validation and test accuracy.
# Use the final validation accuracy from Section 6 and write a short interpretation
# in the markdown cell below.


#### Section 7 — short interpretation

**Write 2–4 sentences:**

1. How close are validation and test accuracy?
2. Does the test result support the estimate obtained on the validation set?
3. Why would repeatedly checking the test set while changing the model make the final estimate less trustworthy?


---
# Part II — Homework: controlled experiments on Fashion-MNIST

The goal is to move from “the network trains” to **evidence-based model choices**.

## Rules for fair comparisons

For every experiment:

1. keep the same train/validation split;
2. reset the random seed before building the model;
3. change only the factor under investigation when possible;
4. use the **validation set**, never the test set, for comparison;
5. record both the configuration and the result.

You do not need exhaustive tuning here. These are small, controlled experiments.


## H1 — Build a reusable configurable MLP

Extend the baseline so that the following choices can be changed without rewriting the training loop:

- hidden-layer width;
- activation function (`ReLU` or `GELU`);
- dropout probability;
- optional Batch Normalization.

The basic structure is provided. Complete only the architecture-building part.


In [ ]:
def make_activation(name):
    if name == "relu":
        return nn.ReLU()
    if name == "gelu":
        return nn.GELU()
    raise ValueError(f"Unknown activation: {name}")


class ConfigurableMLP(nn.Module):
    def __init__(
        self,
        n_inputs=28 * 28,
        hidden1=300,
        hidden2=100,
        n_classes=10,
        activation="relu",
        dropout=0.0,
        batch_norm=False,
    ):
        super().__init__()

        layers = [nn.Flatten()]

        # TODO H1 — append the first hidden block:
        # Linear -> optional BatchNorm1d -> activation -> optional Dropout

        # TODO H2 — append the second hidden block using the same pattern.

        # TODO H3 — append the final Linear classification layer.

        self.net = nn.Sequential(*layers)

    def forward(self, X):
        return self.net(X)


## H2 — A small experiment runner

Most boilerplate is provided. Complete the optimizer choices and use this helper for the comparisons below.


In [ ]:
def run_experiment(config, epochs=3):
    torch.manual_seed(SEED)

    model = ConfigurableMLP(
        hidden1=config.get("hidden1", 300),
        hidden2=config.get("hidden2", 100),
        activation=config.get("activation", "relu"),
        dropout=config.get("dropout", 0.0),
        batch_norm=config.get("batch_norm", False),
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer_name = config.get("optimizer", "sgd")
    lr = config.get("lr", 0.05)
    weight_decay = config.get("weight_decay", 0.0)

    # TODO H4 — create either SGD (with optional momentum) or AdamW.
    # optimizer = ...

    history = []
    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        val = evaluate(model, valid_loader, criterion)
        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val["loss"],
            "val_accuracy": val["accuracy"],
        })

    result = {
        **config,
        "best_val_accuracy": max(row["val_accuracy"] for row in history),
        "final_val_accuracy": history[-1]["val_accuracy"],
    }
    return model, pd.DataFrame(history), result


## H3 — Test the main techniques

Run the following comparisons. For each one, write **2–4 sentences** interpreting what changed and why the result may have changed.

### Experiment A — Activation
Compare `ReLU` and `GELU` with all other settings fixed.

### Experiment B — Optimizer
Compare SGD (with momentum) and AdamW. Choose reasonable learning rates for each optimizer and **state them explicitly**.

### Experiment C — Regularization
Compare the baseline with one or more of:

- Dropout;
- weight decay;
- Batch Normalization.

### Experiment D — Capacity
Change the hidden-layer sizes. Compare parameter count and validation behavior.

Do not run every possible combination here. The goal is to develop hypotheses before the systematic Optuna study.


In [ ]:
experiment_results = []

# Example configuration — keep this as a reference.
baseline_config = {
    "hidden1": 300,
    "hidden2": 100,
    "activation": "relu",
    "dropout": 0.0,
    "batch_norm": False,
    "optimizer": "sgd",
    "lr": 0.05,
    "momentum": 0.0,
    "weight_decay": 0.0,
}

# TODO H5 — define and run a small set of controlled configurations.
# Suggested pattern:
# model_exp, history_exp, result_exp = run_experiment(config, epochs=15)
# experiment_results.append(result_exp)

# TODO H6 — create a DataFrame sorted by validation accuracy.
# fashion_results = ...
# display(fashion_results)


## H4 — Short analysis

Answer directly in the notebook:

1. Which single change produced the clearest improvement over your baseline?
2. Did any technique reduce training loss while not improving validation accuracy? What does that suggest?
3. Which hyperparameters appear most sensitive and deserve systematic tuning?
4. Which choices would you **not** include in the Optuna search, and why?


---
# Part III — Final exercise: CIFAR-10 + Optuna

You will now apply the experimental workflow to a harder dataset.

CIFAR-10 contains 32×32 RGB images from 10 classes. In this exercise you will deliberately use an **MLP** rather than a CNN so that the effect of the techniques studied above is easier to isolate. Flattening images discards spatial structure, so do not expect state-of-the-art accuracy.

Your task is to:

1. prepare a reproducible train/validation/test protocol;
2. define an Optuna search space;
3. train multiple candidate MLPs;
4. use validation accuracy as the objective;
5. prune weak trials;
6. analyze the search results and hyperparameter behavior;
7. rebuild a fresh model using the best configuration;
8. evaluate the test set **once**;
9. save the final model checkpoint.


## C1 — Install and import Optuna

The installation cell is provided because Colab runtimes do not always include Optuna.


In [ ]:
%pip -q install optuna


In [ ]:
import optuna
from optuna.importance import get_param_importances

print("Optuna version:", optuna.__version__)


## C2 — CIFAR-10 data pipeline

The simple preprocessing and split are provided. A moderate subset is used for the search so that experiments remain practical. The final model can be retrained on more data if resources allow.

**Important:** the CIFAR-10 test set is created now but must not be used inside the Optuna objective.


In [ ]:
cifar_transform = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616)),
])

cifar_full_train = torchvision.datasets.CIFAR10(
    root="datasets", train=True, download=True, transform=cifar_transform
)
cifar_test = torchvision.datasets.CIFAR10(
    root="datasets", train=False, download=True, transform=cifar_transform
)

split_generator = torch.Generator().manual_seed(SEED)
cifar_train, cifar_valid = torch.utils.data.random_split(
    cifar_full_train, [45_000, 5_000], generator=split_generator
)

# Search subset: makes repeated trials practical.
SEARCH_TRAIN_SIZE = 15_000
SEARCH_VALID_SIZE = 3_000
cifar_search_train = Subset(cifar_train, range(SEARCH_TRAIN_SIZE))
cifar_search_valid = Subset(cifar_valid, range(SEARCH_VALID_SIZE))

CIFAR_BATCH_SIZE = 256
cifar_train_loader = DataLoader(
    cifar_search_train, batch_size=CIFAR_BATCH_SIZE, shuffle=True, pin_memory=pin
)
cifar_valid_loader = DataLoader(
    cifar_search_valid, batch_size=CIFAR_BATCH_SIZE, shuffle=False, pin_memory=pin
)
cifar_test_loader = DataLoader(
    cifar_test, batch_size=CIFAR_BATCH_SIZE, shuffle=False, pin_memory=pin
)

print("search train:", len(cifar_search_train))
print("search valid:", len(cifar_search_valid))
print("test:", len(cifar_test))
print("classes:", cifar_full_train.classes)


## C3 — Configurable CIFAR-10 MLP

A CIFAR-10 image has `3 × 32 × 32 = 3072` input features after flattening.

Complete the architecture so that Optuna can vary depth, width, activation, dropout, and Batch Normalization.


In [ ]:
class CIFARMLP(nn.Module):
    def __init__(
        self,
        hidden_dims,
        activation="relu",
        dropout=0.0,
        batch_norm=False,
        n_classes=10,
    ):
        super().__init__()
        layers = [nn.Flatten()]
        in_features = 3 * 32 * 32

        # TODO C1 — for each hidden dimension, append:
        # Linear -> optional BatchNorm1d -> activation -> optional Dropout
        # Update `in_features` after each block.

        # TODO C2 — append the final classification layer.

        self.net = nn.Sequential(*layers)

    def forward(self, X):
        return self.net(X)


## C4 — Define the Optuna search space

Search the **main techniques**, not arbitrary parameters. A reasonable search includes:

- number of hidden layers;
- hidden width;
- activation (`relu`, `gelu`);
- dropout;
- Batch Normalization;
- optimizer (`sgd`, `adamw`);
- learning rate;
- weight decay;
- momentum, conditionally when SGD is selected.

Use log-scale sampling for learning rate and weight decay.


In [ ]:
def suggest_config(trial):
    # TODO C3 — sample the number of layers and their widths.
    # n_layers = ...
    # hidden_dims = [...]

    # TODO C4 — sample activation, dropout and batch normalization.
    # activation = ...
    # dropout = ...
    # batch_norm = ...

    # TODO C5 — sample optimizer name, learning rate and weight decay.
    # optimizer_name = ...
    # lr = ...
    # weight_decay = ...

    config = {
        # "hidden_dims": hidden_dims,
        # "activation": activation,
        # "dropout": dropout,
        # "batch_norm": batch_norm,
        # "optimizer": optimizer_name,
        # "lr": lr,
        # "weight_decay": weight_decay,
    }

    # TODO C6 — if SGD is selected, also sample momentum.

    return config


## C5 — Build the optimizer from a sampled configuration

Complete the helper. Keep optimizer-specific arguments conditional.


In [ ]:
def build_optimizer(model, config):
    # TODO C7 — return SGD or AdamW using the values in `config`.
    raise NotImplementedError


## C6 — Optuna objective with intermediate reporting, pruning, and early stopping

Each trial should:

1. sample a configuration;
2. build a fresh model;
3. train for a maximum number of epochs;
4. report validation accuracy after every epoch;
5. allow Optuna to prune poor trials **relative to the other trials**;
6. apply **early stopping inside the trial** when validation accuracy stops improving;
7. return the best validation accuracy reached by the trial.

### Pruning vs early stopping

These mechanisms solve two different problems:

- **Optuna pruning** stops a trial because its intermediate result is not competitive with the other trials seen by the study.
- **Early stopping** stops training inside one trial because that trial's own validation metric has not improved for a fixed patience.

Use validation accuracy for both mechanisms. Do **not** access the test loader here.


In [ ]:
SEARCH_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 1e-3


def objective(trial):
    config = suggest_config(trial)

    torch.manual_seed(SEED)
    model = CIFARMLP(
        hidden_dims=config["hidden_dims"],
        activation=config["activation"],
        dropout=config["dropout"],
        batch_norm=config["batch_norm"],
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = build_optimizer(model, config)

    best_val_acc = 0.0
    epochs_without_improvement = 0

    for epoch in range(SEARCH_EPOCHS):
        # TODO C8 — train one epoch.
        # train_loss = ...

        # TODO C9 — evaluate on cifar_valid_loader.
        # val_metrics = ...
        # val_acc = ...

        # TODO C10 — report the intermediate value to Optuna.
        # trial.report(...)

        # TODO C11 — prune the trial when requested by Optuna.
        # if ...:
        #     raise ...

        # TODO C12 — implement early stopping for this trial.
        # A meaningful improvement is larger than EARLY_STOPPING_MIN_DELTA.
        # If validation accuracy improves:
        #   1. update best_val_acc
        #   2. reset epochs_without_improvement
        # Otherwise:
        #   1. increment epochs_without_improvement
        #   2. stop this trial when patience is reached
        #
        # Suggested pattern:
        # if val_acc > best_val_acc + EARLY_STOPPING_MIN_DELTA:
        #     ...
        # else:
        #     ...
        #
        # if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        #     trial.set_user_attr("early_stopped", True)
        #     trial.set_user_attr("stopped_epoch", epoch + 1)
        #     break

        pass

    return best_val_acc


## C7 — Run the study

Use a seeded TPE sampler and a median-based pruner. Start with a modest number of trials; increase it only if your compute budget allows.

Remember that a trial can now finish in three ways:

- reach `SEARCH_EPOCHS`;
- stop early because its own validation accuracy stopped improving;
- be pruned by Optuna because its intermediate performance is not competitive.


In [ ]:
N_TRIALS = 20

sampler = optuna.samplers.TPESampler(seed=SEED)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    study_name="cifar10_mlp_homework",
)

# TODO C13 — optimize the objective for N_TRIALS.
# study.optimize(...)


## C8 — Analyze the study

A best-trial printout is **not enough**. Your analysis must include:

1. number of completed and pruned trials;
2. number of trials stopped by your early-stopping rule;
3. best validation accuracy;
4. best hyperparameter configuration;
5. a table of the strongest trials;
6. parameter importance;
7. at least **two observations** about hyperparameter behavior;
8. a short comparison of **pruning vs early stopping** observed in your study.

Examples of useful questions:

- Is one optimizer consistently associated with stronger trials?
- Do high learning rates fail more often for one optimizer?
- Is dropout useful across all model sizes or only larger models?
- Does Batch Normalization change the preferred learning-rate range?
- Are deeper/wider MLPs always better on the validation subset?
- Which trials were early-stopped but not pruned, and what does that tell you?


In [ ]:
# TODO C13 — convert trials to a DataFrame and display useful columns.
# trials_df = study.trials_dataframe()
# ...

# TODO C14 — print study.best_value and study.best_params.

# TODO C15 — compute parameter importances.
# importances = get_param_importances(study)
# display(pd.Series(importances, name="importance"))


### Required written analysis

Write **one short paragraph** interpreting the study. Do not merely restate the best parameters. Discuss relationships or trade-offs that you can actually support from your trials.

> **TODO C16 — write your analysis here.**


## C9 — Rebuild and train the final model

After the search is complete:

1. create a **new** model from `study.best_params`;
2. train it again from scratch;
3. use the training and validation data according to a protocol you state explicitly;
4. evaluate the CIFAR-10 test set **once** at the end.

For simplicity, the cells below continue to use the search training/validation loaders. You may optionally retrain on a larger subset if compute permits.


In [ ]:
# TODO C17 — reconstruct a complete config from study.best_params.
# Be careful with conditional parameters such as SGD momentum.
# best_config = ...

# TODO C18 — create a fresh final_model and optimizer.
# torch.manual_seed(SEED)
# final_model = ...
# final_optimizer = ...
# final_criterion = nn.CrossEntropyLoss()

FINAL_EPOCHS = 
final_history = []

# TODO C19 — train the final model and record validation metrics each epoch.


## C10 — Final test evaluation and checkpoint

Only now should you evaluate on `cifar_test_loader`.

Save a checkpoint containing at least:

- `model_state_dict`;
- the selected hyperparameters;
- the class names;
- test accuracy;
- the random seed.

Saving only the full Python model object with `torch.save(model, ...)` is **not** the requested format.


In [ ]:
# TODO C20 — evaluate the final model on the test set exactly once.
# final_test_metrics = ...

checkpoint_path = Path("cifar10_mlp_best.pt")

# TODO C21 — build the checkpoint dictionary and save it with torch.save.
# checkpoint = {...}
# torch.save(checkpoint, checkpoint_path)

# print("saved:", checkpoint_path.resolve())


## C11 — Verify that the checkpoint can be restored

A saved file is useful only if it can be loaded. Reconstruct the architecture from the stored configuration, load the state dictionary, and verify its test accuracy.


In [ ]:
# TODO C22 — load the checkpoint with map_location=device.
# loaded = torch.load(checkpoint_path, map_location=device)

# TODO C23 — rebuild a compatible CIFARMLP, load the state_dict and evaluate it.


---
# Submission checklist

Submit the completed notebook and the generated `cifar10_mlp_best.pt` checkpoint.

Your notebook should clearly contain:

- all completed live TODOs;
- the Fashion-MNIST controlled-experiment table;
- brief interpretations of the Fashion-MNIST experiments;
- the completed Optuna objective and search space;
- working Optuna pruning and per-trial early stopping;
- the Optuna trial-results table;
- parameter-importance analysis;
- a short discussion of hyperparameter behavior;
- final CIFAR-10 validation/test results;
- the saved and successfully reloaded checkpoint.

## Reflection questions

Answer briefly:

1. Why must the test set stay outside the Optuna objective?
2. Why can the best Optuna trial overestimate future performance even though it used a validation set?
3. What is the conceptual difference between Optuna pruning and the early-stopping rule used inside each trial?
4. Which hyperparameter interaction did you find most interesting?
5. Why is an MLP structurally disadvantaged on CIFAR-10 compared with a CNN?
6. If given more compute, what would you change in the experimental protocol before simply increasing `N_TRIALS`?
